In [1]:
import pandas as pd

In [2]:
customers = pd.read_csv("raw/customers.csv")
campaigns =  pd.read_csv("raw/campaigns.csv")
order_items = pd.read_csv("raw/order_items.csv")
orders =  pd.read_csv("raw/orders.csv")
products = pd.read_csv("raw/products.csv")
stores =  pd.read_csv("raw/stores.csv")

In [3]:
tables = {"customers":customers, "campaigns":campaigns, "order_items":order_items, "orders":orders, "products":products, "stores":stores}

for name, table in tables.items():
    print(f"Null Values For {name} Table")
    print(table.isnull().sum())
    print("-" * 30)

Null Values For customers Table
customer_id              0
first_name               0
last_name                0
gender                   0
age_bracket              0
state                  120
city                     0
signup_date              0
acquisition_channel      0
loyalty_tier             0
lifetime_orders          0
dtype: int64
------------------------------
Null Values For campaigns Table
campaign_id         0
campaign_name       0
campaign_type       0
start_date          0
end_date            0
discount_pct        0
channel_scope       0
target_category    14
dtype: int64
------------------------------
Null Values For order_items Table
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
discount_pct     0
line_total       0
dtype: int64
------------------------------
Null Values For orders Table
order_id              0
order_date            0
customer_id           0
channel               0
store_id          18236
payment_method 

In [4]:
customers["state"] = customers["state"].fillna("Unknown")

campaigns["target_category"] = campaigns["target_category"].fillna("General")

orders.loc[orders["channel"] == "Online", "store_id"] = "Online"
orders = orders[orders["store_id"].notna()].copy()

orders["campaign_id"] = orders["campaign_id"].fillna("No Campaign")

products["subcategory"] = products["subcategory"].fillna("Unknown")

tables = {"customers":customers, "campaigns":campaigns, "order_items":order_items, "orders":orders, "products":products, "stores":stores}

In [5]:
for name, table in tables.items():
    print(f"Duplicated Rows For {name} Table")
    print(table.duplicated().sum())
    print("-" * 30)

Duplicated Rows For customers Table
24
------------------------------
Duplicated Rows For campaigns Table
0
------------------------------
Duplicated Rows For order_items Table
1515
------------------------------
Duplicated Rows For orders Table
0
------------------------------
Duplicated Rows For products Table
4
------------------------------
Duplicated Rows For stores Table
0
------------------------------


In [6]:
duplicated_tables = [customers, order_items, products]

for table in duplicated_tables:
    table = table.drop_duplicates(inplace=True)

tables = {"customers":customers, "campaigns":campaigns, "order_items":order_items, "orders":orders, "products":products, "stores":stores}

for name, table in tables.items():
    print(f"Duplicated Rows For {name} Table")
    print(table.duplicated().sum())
    print("-" * 30)

Duplicated Rows For customers Table
0
------------------------------
Duplicated Rows For campaigns Table
0
------------------------------
Duplicated Rows For order_items Table
0
------------------------------
Duplicated Rows For orders Table
0
------------------------------
Duplicated Rows For products Table
0
------------------------------
Duplicated Rows For stores Table
0
------------------------------


In [7]:
customers["state"].unique()
customers["city"].unique()

state_mapping = {
    "CA": "California",
    "GA": "Georgia",
    "FL": "Florida",
    "IL": "Illinois",
    "NY": "New York",
    "TX": "Texas",
    "PA": "Pennsylvania",
    "MO": "Missouri",
    "NJ": "New Jersey",
    "CO": "Colorado",
    "WA": "Washington",
    "NC": "North Carolina",
    "OH": "Ohio",
    "MI": "Michigan",
    "AZ": "Arizona",
    "TN": "Tennessee",
    "VA": "Virginia",
    "MA": "Massachusetts",
    "IN": "Indiana",
    "MD": "Maryland"
}

customers["state"] = customers["state"].str.strip().replace(state_mapping)
customers["city"] = customers["city"].str.strip()

In [8]:
products["unit_price"].unique()
products["unit_price"] = products["unit_price"].astype(str).str.replace("$", "", regex=False).astype(float)

products["brand"].unique()
products["brand"] = products["brand"].str.strip().str.title()

In [9]:
orders["order_date"] = pd.to_datetime(orders["order_date"], format="mixed")

customers["signup_date"] = pd.to_datetime(customers["signup_date"])
products["launch_date"] = pd.to_datetime(products["launch_date"])
stores["open_date"] = pd.to_datetime(stores["open_date"])
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"])
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"])

orders["payment_method"] = orders["payment_method"].str.strip().str.title()

order_totals = order_items.groupby("order_id")["line_total"].sum().round(2)
mismatch = (orders["order_id"].map(order_totals).round(2) != orders["order_total"].round(2)) & orders["order_id"].isin(order_totals.index)
orders["order_total"] = orders["order_id"].map(order_totals).fillna(orders["order_total"]).round(2)

In [10]:
order_items["quantity"] = order_items["quantity"].abs()
order_items = order_items[order_items["product_id"].isin(products["product_id"])]

In [11]:
import os

os.makedirs("clean", exist_ok=True)

clean_tables = {"customers": customers, "campaigns": campaigns, "order_items": order_items, "orders": orders, "products": products, "stores": stores}

for name, table in clean_tables.items():
    table.to_csv(f"clean/{name}.csv", index=False)